# Legacy 10-epoch experiment — preliminary
Old outputs and Results figures are not validated evidence. Shared binary metrics replace the mixed historical definitions. Run from an installed checkout (`python -m pip install -e .`) or add this repository to PYTHONPATH. Final-epoch outer-fold scoring remains exploratory until the nested runner is integrated.


# KSSD UNet++ & UNet3+ Models 5-Fold CV (10 Epochs)
Paper: "KSSD2025: A New Annotated Dataset for Automatic Kidney Stone
Segmentation and Evaluation With Modified U-Net Based Deep Learning Models"

## 1. Setup

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import KFold
from PIL import Image
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
for gpu in tf.config.list_physical_devices("GPU"):
    print(" ", gpu)

## 2. GPU Configuration

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy(["/GPU:0", "/GPU:1"])
else:
    strategy = tf.distribute.get_strategy()

print("Replicas:", strategy.num_replicas_in_sync)

## 3. Hyperparameters

In [ ]:
EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
N_FOLDS = 5
SEED = 42
FILTERS = [16, 32, 64, 128, 256]

## 4. Load Dataset

In [ ]:
DATA_DIR = "/kaggle/input/datasets/murillobouzon/kssd2025-kidney-stone-segmentation-dataset/data"
IMG_DIR = os.path.join(DATA_DIR, "image")
LABEL_DIR = os.path.join(DATA_DIR, "label")

common = sorted(set(os.listdir(IMG_DIR)) & set(os.listdir(LABEL_DIR)))
print(f"Matched pairs: {len(common)}")

X_list, Y_list = [], []
for fname in common:
    img = np.asarray(Image.open(os.path.join(IMG_DIR, fname)).convert("L"), dtype=np.float32) / 255.0
    msk = (np.asarray(Image.open(os.path.join(LABEL_DIR, fname)).convert("L"), dtype=np.float32) > 0).astype(np.float32)
    X_list.append(img[..., np.newaxis])
    Y_list.append(msk[..., np.newaxis])

X = np.array(X_list, dtype=np.float32)
Y = np.array(Y_list, dtype=np.float32)
del X_list, Y_list
gc.collect()

print(f"X: {X.shape}, Y: {Y.shape}")

## 5. Dice Loss & Metrics

In [ ]:
from kssd.tf_metrics import dice_loss, make_metrics


## 6. Model Definitions

### U-Net++, U-Net3+

In [ ]:
!pip install -q keras-unet-collection
from keras_unet_collection import models as kuc

def build_unetpp(input_shape=(512, 512, 1)):
    return kuc.unet_plus_2d(
        input_size=input_shape, filter_num=FILTERS, n_labels=1,
        stack_num_down=2, stack_num_up=2,
        activation="ReLU", output_activation="Sigmoid",
        batch_norm=True, pool=True, unpool=True,
        deep_supervision=False, backbone=None, name="UNetPP",
    )

def build_unet3plus(input_shape=(512, 512, 1)):
    return kuc.unet_3plus_2d(
        input_size=input_shape, n_labels=1,
        filter_num_down=FILTERS,
        filter_num_skip=[22, 22, 22, 22], filter_num_aggregate=110,
        stack_num_down=2, stack_num_up=2,
        activation="ReLU", output_activation="Sigmoid",
        batch_norm=True, pool=True, unpool=True,
        deep_supervision=False, backbone=None, name="UNet3Plus",
    )

MODEL_BUILDERS = {
    "UNetPP": build_unetpp,
    "UNet3Plus": build_unet3plus,
}

print("UNetPP & UNet3Plus ready")

### 6c. Verify all models

In [ ]:
for name, builder in MODEL_BUILDERS.items():
    with strategy.scope():
        m = builder()
    print(f"{name:15s} {m.count_params():>10,} params  |  out={m.output_shape}")
    del m
    tf.keras.backend.clear_session()
    gc.collect()

## 7. Create Folds

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(kf.split(X))
for i, (tr, va) in enumerate(folds, 1):
    print(f"Fold {i}: train={len(tr)}, val={len(va)}")

## 8. Train Models One at a Time

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
MODELS_TO_TRAIN = ["UNetPP", "UNet3Plus"]
all_results = {}

for model_name in MODELS_TO_TRAIN:
    print(f"\n{'#'*60}")
    print(f"# MODEL: {model_name}")
    print(f"{'#'*60}")

    builder = MODEL_BUILDERS[model_name]
    model_histories = {}

    for fold_idx, (train_idx, val_idx) in enumerate(folds, 1):
        print(f"\n--- {model_name} of Fold {fold_idx}/{N_FOLDS} ---")

        train_ds = tf.data.Dataset.from_tensor_slices((X[train_idx], Y[train_idx]))
        train_ds = train_ds.shuffle(len(train_idx), reshuffle_each_iteration=True)
        train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE)

        val_ds = tf.data.Dataset.from_tensor_slices((X[val_idx], Y[val_idx]))
        val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

        with strategy.scope():
            model = builder()
            model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                loss=dice_loss,
                metrics=make_metrics(threshold=0.5),
            )

        history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        model_histories[fold_idx] = history.history

        save_path = f"/kaggle/working/{model_name}_fold{fold_idx}.keras"
        model.save(save_path)
        print(f"Saved: {save_path}")

        del model, train_ds, val_ds
        tf.keras.backend.clear_session()
        gc.collect()

    all_results[model_name] = model_histories
    print(f"\n>>> {model_name} ALL 5 FOLDS DONE <<<")

## 9. Results Summary of UNet++ & UNet3+

In [ ]:
print(f"\n{'='*70}")
print("EXPLORATORY FINAL-EPOCH OUTER-FOLD RESULTS of UNet++ & UNet3+")
print(f"{'='*70}\n")

summary_table = {}

for model_name in MODELS_TO_TRAIN:
    histories = all_results[model_name]
    dice_scores, iou_scores, prec_scores, rec_scores = [], [], [], []

    for fold_idx in range(1, N_FOLDS + 1):
        h = histories[fold_idx]
        dice_scores.append(h["val_hard_mask_dice"][-1])
        iou_scores.append(h["val_hard_mask_iou"][-1])
        prec_scores.append(h["val_hard_mask_precision"][-1])
        rec_scores.append(h["val_hard_mask_recall"][-1])

    summary_table[model_name] = {
        "Dice": (np.mean(dice_scores), np.std(dice_scores)),
        "IoU": (np.mean(iou_scores), np.std(iou_scores)),
        "Precision": (np.mean(prec_scores), np.std(prec_scores)),
        "Recall": (np.mean(rec_scores), np.std(rec_scores)),
    }

    print(f"{model_name}:")
    for fold_idx in range(1, N_FOLDS + 1):
        h = histories[fold_idx]
        print(f"  Fold {fold_idx}: Dice={h['val_hard_mask_dice'][-1]:.4f}  IoU={h['val_hard_mask_iou'][-1]:.4f}")
    print(f"  Mean: Dice={np.mean(dice_scores):.4f}Â±{np.std(dice_scores):.4f}  IoU={np.mean(iou_scores):.4f}Â±{np.std(iou_scores):.4f}")
    print()

print(f"\n{'='*70}")
print(f"{'Model':15s} {'Dice':>14s} {'IoU':>14s} {'Precision':>14s} {'Recall':>14s}")
print(f"{'='*70}")
for model_name, metrics in summary_table.items():
    d = metrics["Dice"]
    i = metrics["IoU"]
    p = metrics["Precision"]
    r = metrics["Recall"]
    print(f"{model_name:15s} {d[0]:.4f}Â±{d[1]:.4f} {i[0]:.4f}Â±{i[1]:.4f} {p[0]:.4f}Â±{p[1]:.4f} {r[0]:.4f}Â±{r[1]:.4f}")

## 10. Training Curves of All Models

In [ ]:
fig, axes = plt.subplots(len(MODELS_TO_TRAIN), 2, figsize=(14, 5 * len(MODELS_TO_TRAIN)))

for row, model_name in enumerate(MODELS_TO_TRAIN):
    histories = all_results[model_name]

    for fold_idx in range(1, N_FOLDS + 1):
        h = histories[fold_idx]
        axes[row, 0].plot(h["val_hard_mask_dice"], label=f"Fold {fold_idx}")
        axes[row, 1].plot(h["val_loss"], label=f"Fold {fold_idx}")

    axes[row, 0].set_title(f"{model_name} Val Dice")
    axes[row, 0].set_xlabel("Epoch")
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].set_title(f"{model_name} Val Loss")
    axes[row, 1].set_xlabel("Epoch")
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)

plt.suptitle("UNet++ & UNet3+ 5-Fold CV Training Curves", fontsize=14)
plt.tight_layout()
plt.savefig("/kaggle/working/all_models_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Sample Predictions of Best Fold of Each Model

In [ ]:
_, last_val_idx = folds[N_FOLDS - 1]
sample_indices = last_val_idx[:4]

fig, axes = plt.subplots(4, len(MODELS_TO_TRAIN) + 2, figsize=(4 * (len(MODELS_TO_TRAIN) + 2), 16))

for i, idx in enumerate(sample_indices):
    axes[i, 0].imshow(X[idx, ..., 0], cmap="gray")
    axes[i, 0].set_title("Input" if i == 0 else "")

    axes[i, 1].imshow(Y[idx, ..., 0], cmap="gray")
    axes[i, 1].set_title("Ground Truth" if i == 0 else "")

    for j, model_name in enumerate(MODELS_TO_TRAIN):
        model_path = f"/kaggle/working/{model_name}_fold{N_FOLDS}.keras"
        m = tf.keras.models.load_model(model_path, compile=False,)
        pred = m.predict(X[idx:idx+1], verbose=0)[0, ..., 0]
        axes[i, j + 2].imshow(pred >= 0.5, cmap="gray")
        axes[i, j + 2].set_title(model_name if i == 0 else "")
        del m
        tf.keras.backend.clear_session()

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Sample Predictions of UNet++ & UNet3+ (Fold 5)", fontsize=14)
plt.tight_layout()
plt.savefig("/kaggle/working/all_models_predictions.png", dpi=150, bbox_inches="tight")
plt.show()